# Kundensegmentierung mit Machine Learning (K-Means Clustering) - Woche 3

In dieser Woche wenden wir unüberwachtes maschinelles Lernen (Unsupervised Machine Learning) an, um unsere Kundenbasis in verschiedene **Cluster (Segmente)** zu unterteilen. So können wir später gezielte Gutscheine verteilen.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

sns.set_theme(style="whitegrid")

# 1. Wir laden die aus Woche 2 generierten Nutzerdaten
df = pd.read_csv('../data/user_base.csv')
print(f"Anzahl der initialen User: {len(df)}")

## 1. Feature Auswahl und Skalierung

K-Means verwendet Distanzen (Euklidische Distanz), um zu berechnen, wie ähnlich Nutzer zueinander sind. Jemand, der 5.000$ ausgegeben hat, hätte ohne Skalierung ein viel größeres statistisches Gewicht als jemand mit "4 total_trips".
Daher benutzen wir den **StandardScaler**, der jedes Feature so anpasst, dass der Mittelwert 0 und die Varianz 1 beträgt.

In [ ]:
features = [
    'total_sessions', 
    'cancellation_rate', 
    'window_shopping_rate',
    'discount_affinity',
    'avg_page_clicks',
    'total_trips', 
    'total_spent',
    'avg_nights_hotel',
    'avg_booking_lead_time',
    'business_trip_ratio', 
    'family_trip_ratio',
    'age',
    'is_married',
    'has_kids'
]

X = df[features].fillna(0).copy()

# Skalieren der Daten
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Zeige die ersten paar Zeilen als Pandas DataFrame
X_scaled_df = pd.DataFrame(X_scaled, columns=features)
display(X_scaled_df.head())

## 2. Bestimmen der optimalen Cluster-Anzahl (`k`)
Meist wird hierfür die *Elbow Methode* oder der *Silhouette Score* verwendet. Wir versuchen k (Anzahl Cluster) im Intervall von 2 bis z.B. 8.

In [ ]:
inertias = []
K_range = range(2, 9)

# Sample für den Silhouette Score (da 94k Nutzer etwas lange dauern könnten)
sample_idx = np.random.choice(X_scaled.shape[0], size=10000, replace=False)
X_sample = X_scaled[sample_idx, :]

sil_scores = []

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertias.append(kmeans.inertia_)
    
    # Silhouette Score mit dem Sample berechnen
    clusters_sample = kmeans.predict(X_sample)
    score = silhouette_score(X_sample, clusters_sample)
    sil_scores.append(score)
    print(f"k={k}: Silhouette Score = {score:.4f}")

# Plotting (Elbow Method)
plt.figure(figsize=(10, 4))
plt.plot(K_range, inertias, marker='o')
plt.title('Elbow Method (Inertia)')
plt.xlabel('Anzahl der Cluster (k)')
plt.ylabel('Sum of Squared Distances')
plt.show()

# Plotting (Silhouette Scores)
plt.figure(figsize=(10, 4))
plt.plot(K_range, sil_scores, marker='s', color="orange")
plt.title('Silhouette Score')
plt.xlabel('Anzahl der Cluster (k)')
plt.ylabel('Score')
plt.show()

## 3. Anwenden von K-Means (z.B. k=4)
Wählen wir beispielsweise $k=4$ (da hier oft der Sweet-Spot zwischen Interpretierbarkeit und Clustering-Qualität liegt).

In [ ]:
best_k = 4
print(f"Wir wählen {best_k} Cluster und weisen jedem User einen Cluster zu.")

kmeans_final = KMeans(n_clusters=best_k, random_state=42, n_init=10)
cluster_labels = kmeans_final.fit_predict(X_scaled)

# An das ursprüngliche df anfügen
df['cluster'] = cluster_labels

print(df['cluster'].value_counts())

## 4. PCA (Principal Component Analysis zur Visualisierung)
Menschen können keine n-Dimensionalen Räume sehen (wir haben oben 9 Features / Dimensionen!). Um die Cluster 2D plotten zu können, nutzen wir PCA, um die Dimensionalität auf die 2 wichtigsten kombinierten Achsen zu reduzieren.

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

pca_df = pd.DataFrame(data=X_pca, columns=['PC1', 'PC2'])
pca_df['Cluster'] = cluster_labels

plt.figure(figsize=(10, 8))
sns.scatterplot(
    x='PC1', y='PC2', 
    hue='Cluster', 
    data=pca_df, 
    palette='tab10', 
    s=50, alpha=0.6
)
plt.title('2D PCA Visualization der 4 Kunden-Segmente')
plt.show()

## 5. Interpretation der Cluster
Nun vergleichen wir die reinen Durchschnittswerte jedes Features pro Cluster. Nur so erkennen wir, wen K-Means überhaupt gefunden hat! Wir gruppieren am originalen DataFrame!

In [ ]:
cluster_analysis = df.groupby('cluster')[features].mean().round(2)
display(cluster_analysis)

# Hilft uns eine Heatmap, um das auf einen Blick zu verstehen?
# Wir können die StandardScaler Features mitteln, um zu sehen wie weit eine Gruppe über oder unter dem globalen Durchschnitt liegt.

cluster_z_scores = pd.DataFrame(X_scaled, columns=features)
cluster_z_scores['cluster'] = cluster_labels
cluster_z_scores_grouped = cluster_z_scores.groupby('cluster').mean()

plt.figure(figsize=(12, 6))
sns.heatmap(cluster_z_scores_grouped, annot=True, cmap='coolwarm', center=0)
plt.title('Feature Ausprägung pro Cluster (Z-Scores)')
plt.show()

So können wir in Woche 3 die *Bedeutung* der Gruppen ablesen (z.B. "Cluster 1 ist der Window Shopper, sehr hohe window_shopping_rate. Cluster 2 liebt es, Familienreisen zu buchen, hohe family_trip_ratio.")

Wir speichern das Segmentierungs-Ergebnis abschließend im `data/` Ordner.

In [ ]:
df.to_csv('../data/user_segments.csv', index=False)
print("✅ Segmentierte Kunden (user_segments.csv) gespeichert!")

## Verteilung der Cluster (Balkendiagramm)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6))
ax = sns.countplot(x='cluster', data=df, hue='cluster', palette='viridis', legend=False)
plt.title('K-Means Cluster Gruppen Verteilung', fontsize=16)
plt.xlabel('Cluster Segment')
plt.ylabel('Anzahl der User')

# Werte über die Balken schreiben
for p in ax.patches:
    ax.annotate(f"{int(p.get_height())}", 
                (p.get_x() + p.get_width() / 2., p.get_height()), 
                ha='center', va='bottom', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()